# Quantization Experiment: Token Vector Serialization

A quick experiment to measure the size and latency impact of different 
serialization strategies for ColBERT token vectors.

This is a standalone experiment outside the main guide — the goal is to 
understand whether quantizing token vectors before JSON serialization would 
meaningfully reduce the payload overhead observed in the late interaction guide.

Three approaches compared:
- **float32 JSON** — what the guide uses today
- **int8 JSON** — quantized to 8-bit integers before serialization  
- **binary float32** — what native multi-vector support in turbopuffer would look like

The hypothesis: JSON string overhead dominates over float precision. 
Quantization helps when storing binary but is less effective when the 
bottleneck is the JSON format itself.

In [2]:
import os
import json
import numpy as np
import turbopuffer
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-central1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-late-interaction')

In [3]:
# fetch one document with token_vectors
result = ns.query(
    rank_by=["id", "asc"],
    limit=1,
    include_attributes=["token_vectors"]
)

sample_tokens = np.array(json.loads(result.rows[0].token_vectors))
print(f"Token matrix shape: {sample_tokens.shape}")

Token matrix shape: (26, 128)


In [4]:
# approach 1 - raw JSON float32 (what we did)
json_size = len(json.dumps(sample_tokens.tolist()).encode('utf-8'))

# approach 2 - quantize to int8 then JSON
int8_tokens = (sample_tokens * 127).astype(np.int8)
int8_size = len(json.dumps(int8_tokens.tolist()).encode('utf-8'))

# approach 3 - binary (what native multi-vector would do)
binary_size = sample_tokens.astype(np.float32).nbytes

print(f"float32 JSON:  {json_size} bytes")
print(f"int8 JSON:     {int8_size} bytes")
print(f"binary float32: {binary_size} bytes")
print(f"int8 vs float32 JSON savings: {(1 - int8_size/json_size)*100:.1f}%")
print(f"binary vs float32 JSON savings: {(1 - binary_size/json_size)*100:.1f}%")

float32 JSON:  72204 bytes
int8 JSON:     12824 bytes
binary float32: 13312 bytes
int8 vs float32 JSON savings: 82.2%
binary vs float32 JSON savings: 81.6%


## Quantization Experiment Results

### Hypothesis
JSON string overhead dominates over float precision. Quantization helps when 
storing binary but is less effective when the bottleneck is the JSON format itself.

### Results

| Approach | Size (1 document) | vs float32 JSON |
|---|---|---|
| float32 JSON (current) | 72,204 bytes | baseline |
| int8 JSON (quantized) | 12,824 bytes | 82.2% smaller |
| binary float32 (native) | 13,312 bytes | 81.6% smaller |

### What This Means

**The hypothesis was wrong.** Quantization helps significantly even in JSON format, 
not just in binary storage.

Serializing float32 values like `0.0023847` as JSON strings is extremely wasteful. 
Quantizing to int8 first converts those to small integers like `0` or `1` — 
much shorter strings, 82% smaller payload.

**At query time**, fetching token vectors for 100 candidates currently sends:
- float32 JSON: ~7.2MB per query (100 × 72KB)
- int8 JSON: ~1.3MB per query (100 × 12.8KB)

That 80% payload reduction would bring ColBERT fetch latency much closer to 
dense retrieval without any turbopuffer changes needed.

**Binary float32 is almost identical in size to int8 JSON (13.3KB vs 12.8KB).** 
This confirms that native multi-vector support in turbopuffer storing binary would 
achieve similar payload savings to int8 quantization — but with the added benefit 
of server-side MaxSim scoring eliminating the fetch entirely.

### Conclusion

int8 quantization before JSON storage is a meaningful V2 optimization available 
today — no turbopuffer changes required. It would reduce namespace storage from 
~19.63MB to ~3.5MB and cut payload fetch latency by ~80%.

The D2 roadmap suggestion (native multi-vector + server-side MaxSim) remains the 
complete solution — it eliminates the payload fetch entirely rather than just 
shrinking it.